# Medical Entity Extraction from Clinical Notes

## Introduction
In the healthcare domain, unlocking insights from unstructured clinical notes is a massive challenge. Doctors write free-form notes that contain valuable data about symptoms, diagnoses, and treatment plans.

In this cookbook, we will use **Nugen's API** to build a pipeline that:
1. Analyzes a raw clinical note.
2. Extracts structured entities (Symptoms, Medications, Diagnosis).
3. Formats the output as strict JSON for database insertion.
4. Identifies and redacts PII (Personally Identifiable Information) to ensure privacy compliance.

---

## 1. Setup and Configuration

First, we need to install the necessary libraries and set up our environment.
Make sure you have created a `.env` file in the root directory with your `NUGEN_API_KEY`.

In [ ]:
!pip install requests python-dotenv

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

NUGEN_API_KEY = os.getenv("NUGEN_API_KEY")
API_URL = "https://api.nugen.in/inference/chat/completions"

if not NUGEN_API_KEY:
    raise ValueError("Please set NUGEN_API_KEY in your .env file")

print("API Key successfully loaded.")

## 2. Define Helper Functions

We'll define a simple wrapper function to interact with the Nugen API. This keeps our code clean and reusable.

In [ ]:
def get_nugen_response(system_prompt, user_prompt, temperature=0.1):
    headers = {
        "Authorization": f"Bearer {NUGEN_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "model": "nugen-flash-instruct", # Using the fast, instruction-tuned model
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": 1024
    }
    
    try:
        response = requests.post(API_URL, headers=headers, json=payload)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content']
    except Exception as e:
        print(f"Error: {e}")
        return None

## 3. Sample Data

Let's define a synthetic clinical note. 
> **Note**: This is fake patient data generated for demonstration purposes.

In [ ]:
clinical_note = """
Patient: Sarah Connor (DOB: 05/12/1984)
Date of Visit: Oct 24, 2024
ID: 9982-11

Subjective: Patient presents with a 3-day history of productive cough and fever reaching 102F. Reports shortness of breath when climbing stairs. Mentions a history of mild asthma.

Objective: BP 120/80, HR 98, Temp 101.5F. Lungs: Wheezing heard in lower left lobe.

Assessment: Likely Acute Bronchitis. Rule out Pneumonia.

Plan: 
1. Prescribed Amoxicillin 500mg three times daily for 7 days.
2. Use Albuterol inhaler 2 puffs every 4 hours as needed for wheezing.
3. Follow up in 5 days if symptoms do not improve.
"""

## 4. Structured Information Extraction

We want to convert this unstructured text into a JSON object with specific fields:
- `patient_id`
- `diagnosis` (list)
- `symptoms` (list)
- `medications` (list of objects with name, dosage, frequency)

We will craft a system prompt that explicitly defines the output schema.

In [ ]:
extraction_system_prompt = """
You are an expert medical AI assistant. Your task is to extract structured information from clinical notes.
Output must be valid JSON ONLY. Do not include markdown formatting or explanations.

Schema:
{
  "patient_id": "string",
  "diagnosis": ["string"],
  "symptoms": ["string"],
  "medications": [
    {
      "name": "string",
      "dosage": "string",
      "frequency": "string"
    }
  ]
}
"""

response_json = get_nugen_response(extraction_system_prompt, clinical_note)

print("Raw Response:")
print(response_json)


In [ ]:
# Parse and display the JSON nicely
try:
    parsed_data = json.loads(response_json)
    print(json.dumps(parsed_data, indent=2))
except json.JSONDecodeError:
    print("Failed to parse JSON. Raw output may contain text.")

## 5. Privacy and PII Redaction

For compliance (HIPAA/GDPR), we often need to share notes without revealing patient identity. Let's ask the model to redact names, dates, and IDs, replacing them with `[REDACTED]`.

In [ ]:
redaction_system_prompt = """
You are a privacy compliance officer. 
Identify all Personally Identifiable Information (PII) such as Names, Dates, Dates of Birth, and IDs in the text. 
Replace them with the placeholder '[REDACTED]'.
Return the modified text only.
"""

redacted_note = get_nugen_response(redaction_system_prompt, clinical_note)

print("Original Note Snippet:", clinical_note[:100].replace('\n', ' '), "...")
print("\nRedacted Note:")
print(redacted_note)

## Conclusion

In this cookbook, we demonstrated how Nugen's models can be used to:
1. **Structure Data**: Transforming messy text into queryable JSON.
2. **Protect Privacy**: Automatically scrubbing PII from sensitive documents.

This workflow serves as a foundational block for building automated health record systems, patient analytics dashboards, or research databases.

### Next Steps
- Try running this on a batch of files.
- Experiment with the `nugen-flash-embed` model to cluster similar patient cases.
